# Giai đoạn 3 — Feature engineering (bản FIX, không leakage).

Giai đoạn 3 — Feature engineering (bản FIX, không leakage).
Input : datas/02_cleaned/cleaned.csv
Output: datas/03_features/features.csv + feature_list.json
- is_weekend, cyclic hr/mnth/weekday (sin/cos)
- rush theo DOMAIN (sáng 7-9, chiều 17-19, trưa 12-14), KHÔNG dò peak từ target
- time_period (Night/Morning/Afternoon/Evening/Late Night)
- temp_hum_interaction (không tạo temp_squared/hum_squared vì Poly degree=2 đã sinh)
- comfort = temp*(1-hum), is_bad_weather = weathersit>=3
- lag1/lag24 + roll3/roll24 cho casual/registered/cnt (shift quá khứ -> không leakage, NaN đầu chuỗi điền 0)
- Bỏ atemp (cộng tuyến ~0.99 với temp)
Chạy: python src/03_features.py

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
# Cấu hình đường dẫn (inline để notebook chạy độc lập, không cần config.py)
ROOT = Path.cwd()
if not (ROOT / "datas").exists() and (ROOT.parent / "datas").exists():
    ROOT = ROOT.parent  # khi kernel chạy từ trong thư mục src/
RAW_CSV = ROOT / "datas" / "hour.csv"
EDA_DIR = ROOT / "datas" / "01_eda"
CLEANED_DIR = ROOT / "datas" / "02_cleaned"
FEATURES_DIR = ROOT / "datas" / "03_features"
SPLIT_DIR = ROOT / "datas" / "04_split"
MODELS_DIR = ROOT / "datas" / "05_models"
EVAL_DIR = ROOT / "datas" / "06_evaluation"
CAT_COLS = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "time_period"]
TEST_SIZE = 0.2
RANDOM_STATE = 42


In [2]:
def get_time_period(hr: int) -> str:
    hr = int(hr)
    if 0 <= hr <= 5:
        return "Night"
    if 6 <= hr <= 11:
        return "Morning"
    if 12 <= hr <= 16:
        return "Afternoon"
    if 17 <= hr <= 21:
        return "Evening"
    return "Late Night"


In [3]:
def main():
    FEATURES_DIR.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(CLEANED_DIR / "cleaned.csv")
    df["dteday"] = pd.to_datetime(df["dteday"])

    df["is_weekend"] = df["weekday"].astype(int).isin([0, 6]).astype(int)

    df["hr_sin"] = np.sin(2 * np.pi * df["hr"].astype(int) / 24)
    df["hr_cos"] = np.cos(2 * np.pi * df["hr"].astype(int) / 24)
    df["mnth_sin"] = np.sin(2 * np.pi * df["mnth"].astype(int) / 12)
    df["mnth_cos"] = np.cos(2 * np.pi * df["mnth"].astype(int) / 12)
    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"].astype(int) / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"].astype(int) / 7)

    df["morning_rush"] = df["hr"].astype(int).isin([7, 8, 9]).astype(int)
    df["evening_rush"] = df["hr"].astype(int).isin([17, 18, 19]).astype(int)
    df["midday_rest"] = df["hr"].astype(int).isin([12, 13, 14]).astype(int)
    wd = df["workingday"].astype(int)
    df["workingday_morning_rush"] = ((wd == 1) & (df["morning_rush"] == 1)).astype(int)
    df["workingday_evening_rush"] = ((wd == 1) & (df["evening_rush"] == 1)).astype(int)
    df["non_workingday_midday"] = ((wd == 0) & (df["midday_rest"] == 1)).astype(int)

    df["time_period"] = df["hr"].apply(get_time_period).astype("category")
    df["temp_hum_interaction"] = df["temp"] * df["hum"]
    # Cải tiến: chỉ số thoải mái nhiệt (nóng + ẩm -> giảm thuê)
    df["comfort"] = df["temp"] * (1 - df["hum"])
    # Cải tiến: cờ thời tiết xấu / mưa (weathersit 3-4), thay vì one-hot mù
    df["is_bad_weather"] = (df["weathersit"].astype(int) >= 3).astype(int)
    if "atemp" in df.columns:  # giảm đa cộng tuyến temp/atemp
        df = df.drop(columns=["atemp"])

    # Cải tiến: lag/rolling CHỈ dùng quá khứ (shift) -> không leakage.
    # df đã sort (dteday, hr) từ giai đoạn 2 nên thứ tự giờ liên tục.
    df = df.sort_values(["dteday", "hr"]).reset_index(drop=True)
    for target in ["casual", "registered", "cnt"]:
        df[f"{target}_lag1"] = df[target].shift(1)
        df[f"{target}_lag24"] = df[target].shift(24)  # cùng giờ hôm qua
        df[f"{target}_roll3"] = df[target].shift(1).rolling(3, min_periods=1).mean()
        df[f"{target}_roll24"] = df[target].shift(1).rolling(24, min_periods=1).mean()
    # Đầu chuỗi chưa có quá khứ -> điền 0 (trung tính với count, không dùng stat toàn cục)
    lag_cols = [c for c in df.columns if "_lag" in c or "_roll" in c]
    df[lag_cols] = df[lag_cols].fillna(0)

    for c in CAT_COLS:  # đảm bảo category được giữ
        if c in df.columns:
            df[c] = df[c].astype("category")

    num_cols = [c for c in df.select_dtypes(include="number").columns
                if c not in ("instant", "cnt", "casual", "registered")]
    cat_cols = [c for c in CAT_COLS if c in df.columns]

    df.to_csv(FEATURES_DIR / "features.csv", index=False)
    (FEATURES_DIR / "feature_list.json").write_text(json.dumps(
        {"cat_cols": cat_cols, "num_cols": num_cols}, indent=2), encoding="utf-8")
    print(f"cat: {cat_cols}\nnum: {num_cols}")
    print(f"OK -> {FEATURES_DIR / 'features.csv'} ({len(df)} dòng)")


In [4]:
if __name__ == "__main__":
    main()


cat: ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'time_period']
num: ['temp', 'hum', 'windspeed', 'is_weekend', 'hr_sin', 'hr_cos', 'mnth_sin', 'mnth_cos', 'weekday_sin', 'weekday_cos', 'morning_rush', 'evening_rush', 'midday_rest', 'workingday_morning_rush', 'workingday_evening_rush', 'non_workingday_midday', 'temp_hum_interaction', 'comfort', 'is_bad_weather', 'casual_lag1', 'casual_lag24', 'casual_roll3', 'casual_roll24', 'registered_lag1', 'registered_lag24', 'registered_roll3', 'registered_roll24', 'cnt_lag1', 'cnt_lag24', 'cnt_roll3', 'cnt_roll24']
OK -> /mnt/d/Documents/UIT/HK2/CKIE313/datas/03_features/features.csv (17379 dòng)
